# Experiment 1: Hadoop HDFS Installation and Commands

## 1. Install Java + SSH

In [ ]:
%%bash
sudo apt update
sudo apt install -y openjdk-11-jdk ssh
sudo systemctl start ssh

## 2. Passwordless SSH

In [ ]:
%%bash
ssh-keygen -t rsa -P "" -f ~/.ssh/id_rsa
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
chmod 0600 ~/.ssh/authorized_keys
ssh localhost        # type yes, then: exit

## 3. Download + install Hadoop

In [ ]:
%%bash
wget https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
tar -xzf hadoop-3.3.6.tar.gz
sudo mv hadoop-3.3.6 /usr/local/hadoop
sudo chown -R $USER:$USER /usr/local/hadoop

## 4. Environment variables (saved permanently)

In [ ]:
%%bash
cat >> ~/.bashrc << 'EOF'
export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
export HADOOP_HOME=/usr/local/hadoop
export HADOOP_MAPRED_HOME=$HADOOP_HOME
export PATH=$PATH:$HADOOP_HOME/bin:$HADOOP_HOME/sbin
EOF
source ~/.bashrc
echo "export JAVA_HOME=$JAVA_HOME" >> $HADOOP_HOME/etc/hadoop/hadoop-env.sh
hadoop version

## 5. Config files

In [ ]:
%%bash
cd $HADOOP_HOME/etc/hadoop

cat > core-site.xml << 'EOF'
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>
EOF

cat > hdfs-site.xml << 'EOF'
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
</configuration>
EOF

cat > mapred-site.xml << 'EOF'
<configuration>
  <property><name>mapreduce.framework.name</name><value>yarn</value></property>
  <property><name>mapreduce.application.classpath</name>
    <value>$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/*:$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/lib/*</value></property>
</configuration>
EOF

cat > yarn-site.xml << 'EOF'
<configuration>
  <property><name>yarn.nodemanager.aux-services</name><value>mapreduce_shuffle</value></property>
</configuration>
EOF
cd ~

## 6. Format + start
Expected `jps` output: NameNode, DataNode, SecondaryNameNode, ResourceManager, NodeManager.

Web UI: http://localhost:9870 (HDFS), http://localhost:8088 (YARN)

In [ ]:
%%bash
hdfs namenode -format
start-dfs.sh
start-yarn.sh
jps

## 7. HDFS commands

In [ ]:
%%bash
echo "Hello Hadoop. This is BDA practical." > sample.txt
hdfs dfs -mkdir -p /user/student/data
hdfs dfs -put sample.txt /user/student/
hdfs dfs -copyFromLocal sample.txt /user/student/data/sample2.txt
hdfs dfs -ls /user/student
hdfs dfs -ls -R /user
hdfs dfs -cat /user/student/sample.txt
hdfs dfs -tail /user/student/sample.txt
hdfs dfs -touchz /user/student/empty.txt
hdfs dfs -appendToFile sample.txt /user/student/sample.txt
hdfs dfs -get /user/student/sample.txt ~/sample_copy.txt
hdfs dfs -copyToLocal /user/student/data/sample2.txt ~/sample2_copy.txt
hdfs dfs -cp /user/student/sample.txt /user/student/data/sample_cp.txt
hdfs dfs -mv /user/student/data/sample_cp.txt /user/student/sample_moved.txt
hdfs dfs -du -h /user/student
hdfs dfs -count /user/student
hdfs dfs -df -h
hdfs dfs -stat "%n %b %r" /user/student/sample.txt
hdfs dfs -setrep 1 /user/student/sample.txt
hdfs dfs -chmod 755 /user/student/sample.txt
hdfs dfs -rm /user/student/sample_moved.txt
hdfs dfs -rm -r /user/student/data

## 8. Stop

In [ ]:
%%bash
stop-yarn.sh
stop-dfs.sh